# Session 6: Classifying and evaluating manifesto statements

Use the six-label starter prompt from Session 5 to classify a collection of
sentences. Inspect a confusion matrix, compute precision, recall and F1, and
read the texts behind selected errors.

## Setup

In [ ]:
import os
from pathlib import Path
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

from utils import (
    LABELS, NO_PREDICTION, STARTER_PROMPT,
    classify_text,
    classify_texts_concurrently,
)

from IPython.display import HTML, display
print_html = lambda x, index=False, **kwargs: display(HTML(x.to_html(index=index, **kwargs)))

load_dotenv()

In [ ]:
root_folder = Path('../../')

data_dir = root_folder / "data" / "examples" / "cap"

We use the course Python 3.13 `.venv` kernel and the same `.env` file as in Session 5.
The [Session 5 notebook](prompting_basics.ipynb) shows how to set it up.

In [ ]:
api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
if not api_key or api_key == "PASTE_THE_COURSE_KEY_HERE":
    raise ValueError("Add the course key to OPENROUTER_API_KEY in the course .env file.")
base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

client = OpenAI(base_url=base_url, api_key=api_key, timeout=30, max_retries=1)

In [ ]:
MODEL_ID = "google/gemma-4-26b-a4b-it"
print("Model:", MODEL_ID)
print(STARTER_PROMPT)

## 1. Load the data

The data contain 150 familiar manifesto sentences from Block 2. 
The course review marked 138 for today’s evaluation, with reference labels for *Security/defence*, *International trade* and *Aid*.
The other twelve remain in the CSV for discussion.
<!-- The prompt still allows all six labels. -->

In [ ]:
df = pd.read_csv(root_folder / "data" / "examples" / "cap" / "block3_classification_live.csv")
df = df.loc[
    df["include_in_three_class_evaluation"],
    ["text_id", "text", "reference_label"],
]

In [ ]:
reference_labels = ["Security/defence", "International trade", "Aid"]
df["reference_label"].value_counts().reindex(reference_labels)

## 2. Classify one sentence

Note that we re-use a function like the `classify_text` function we defined in the notebook for session 5.
But this time we use the version defined in [`utils.py`](utils.py):

In [ ]:
from utils import classify_text

example = df.iloc[0]
print(example["text"])

prediction = classify_text(client, STARTER_PROMPT, example["text"], model_id=MODEL_ID)
print("Model label:", prediction)
print("Reference label:", example["reference_label"])

## 3. Apply the same prompt to multiple sentences

`classify_texts_concurrently()` sends one request per text and runs a few requests in parallel.
It returns one result per text, in input order, so we can attach the results to the original rows.

We set `do_subsample = True` to sample 5 examples per label class.
You can set it to `False` to process all 138 texts.
<!-- **_Note:_** Use **Interrupt Kernel** if you need to stop the batch. Completed results are
kept and unfinished rows are marked `interrupted`. Rerun the batch before scoring. -->

In [ ]:
do_subsample = False # NOTE: set to False if you want to classify all texts in `df`
sample = df.copy() if not do_subsample else df.groupby(
    "reference_label", sort=False
).head(5)

In [ ]:
from utils import classify_texts_concurrently

predictions = classify_texts_concurrently(
    client=client,
    prompt=STARTER_PROMPT,
    texts=sample["text"],
    model_id=MODEL_ID,
    max_concurrent=8
)
results = sample.reset_index(drop=True).join(predictions)

In [ ]:
results["status"].value_counts()

In [ ]:
# show any errors in the classification process
if results["status"].ne("ok").any():
    print_html(results.loc[results["status"].ne("ok"), ["text_id", "status", "detail"]])

In [ ]:
# show the classification results for the texts that were successfully classified
completed = results["status"]=="ok"
print_html(results.loc[completed, ["text", "reference_label", "predicted_label"]])

## 4. Exercise 1

**Discuss:** How could we _quantitatively_ evaluate the model's performance on the full set of 138 statements?

## 5. The confusion matrix

A **confusion matrix** cross-tabulates predicted and reference labels.
It allows us to see how often and how predictions deviate from the reference labels.

Here, we show the reference labels in the rows are columns are the model outputs (predictions).

- Read across arow to see what happened to sentences with that reference label. 
- Read down a column to see which references the model assigned to that prediction.

Note that all six categories permitted by the `STARTER_PROMPT` are shown.
If a request failed or an answer could not be used as a label, it would appear under *No valid prediction* and
counts as a missed reference label.
<!-- Use the status counts to distinguish API errors from classification mistakes. -->

In [ ]:
prediction_columns = LABELS.copy()
if results["predicted_label"].eq(NO_PREDICTION).any():
    prediction_columns.append(NO_PREDICTION)

matrix = pd.crosstab(results["reference_label"], results["predicted_label"])
matrix = matrix.reindex(index=reference_labels, columns=prediction_columns, fill_value=0)
print_html(matrix.rename_axis(index="Reference label", columns=""), index=True)

**Read the _Aid_ row.** How many Aid sentences received the correct label?
Did the model choose a label outside the three reference classes? Pick one
such cell if it is populated and describe what its count means.

## 6. Compute precision, recall and F1 for one class

For a single class:

- **Precision** is the fraction of predictions for that class that were correct.
- **Recall** is the fraction of references for that class that the model found.
- **F1** combines the two as a harmonic mean. It is low if either value is low.

Find the row and column totals for *Aid* in the matrix before running the cell.
An Aid reference predicted as Diplomacy is a missed Aid case and lowers Aid recall.

In [ ]:
focal_label = "Aid"
true_positives = matrix.loc[focal_label, focal_label]
predicted_as_label = matrix[focal_label].sum()
referenced_as_label = matrix.loc[focal_label].sum()

precision = true_positives / predicted_as_label if predicted_as_label else 0.0
recall = true_positives / referenced_as_label if referenced_as_label else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"{focal_label}: precision={precision:.3f}, recall={recall:.3f}, F1={f1:.3f}")

<span style="color: violet">**TODO**</span>: Compute the values for another focal label class (e.g., "Security/defence")

The function `f1_score` from the `sklearn`'s `metrics` module is a conventient helper to compute the F1 score

When we set `average=None`, we get the category-specific F1 scores:

In [ ]:
from sklearn.metrics import f1_score

class_f1 = f1_score(
    y_true=results["reference_label"],
    y_pred=results["predicted_label"],
    labels=reference_labels, 
    average=None, 
    zero_division=0,
)
display(pd.Series(class_f1, index=reference_labels, name="F1").round(3))

**Macro-F1** is the unweighted mean of the three class-specific F1 scores.
We average over the classes with reference examples. Predictions in the other
columns still reduce recall for the reference classes. If a metric has a zero
denominator, the code reports zero.

In [ ]:
print(f"Macro-F1 across the three reference classes: {class_f1.mean():.3f}")

# NOTE: this gives the same result:
f1_score(
    y_true=results["reference_label"],
    y_pred=results["predicted_label"],
    labels=reference_labels, 
    average='macro', 
    zero_division=0,
)

Another function from `sklearn`'s `metrics` module helps us to view all results at once: `classification_report`

In [ ]:
from sklearn.metrics import classification_report

class_report = classification_report(
    y_true=results["reference_label"],
    y_pred=results["predicted_label"],
    labels=reference_labels, 
    zero_division=0,
)
print(class_report)

**_Note:_** See the explainer notebook [`classification_evaluation_metrics_explained.ipynb`](classification_evaluation_metrics_explained.ipynb) for a more thorough explanation.

## 7. Exercise 2

In [ ]:
# NOTE: a fictious classification results table for practicing results interpretation

classes = ["Security/defence", "International trade", "Aid"]
precision = [0.8, 0.5, 0.9]
recall = [0.7, 0.6, 0.8]
f1 = [2 * p * r / (p + r) for p, r in zip(precision, recall)]

macro_precision = sum(precision) / len(precision)
macro_recall = sum(recall) / len(recall)
macro_f1 = sum(f1) / len(f1)

# format the results in a table
results_table = pd.DataFrame({
    "Class": [*classes, "<i>Macro-average</i>"],
    "Precision": [*precision, macro_precision],
    "Recall": [*recall, macro_recall],
    "F1": [*f1, macro_f1],
})
print_html(results_table, escape=False, index=False, float_format="{:.3f}".format)

<span style="color: violet">**TODO**</span>: Interpret the fictious classification metric estimates above and answer the following questions:

1. What share of _Security/defence_ cases in the reference data are correctly classified?
2. What share of cases classified as _Aid_ are correctly classified?
3. For which class is the LLM most reliable?
4. What information do the macro-average metrics provide? what do they "hide"?
5. interpreting precision--recall trade-off
    - which category/categories assigns the classifier more often than they occur in the reference data?
    - which category/categories assigns the classifier less often than they occur in the reference data?

## 8. Look at the sentences behind the errors

Inspect the model's classification errors, if available.

What in the text or instructions might have led to the prediction?

A difference can point to an unclear boundary, a rule the model missed, or a sentence that two coders might reasonably read differently.
Discuss a rule you could add?

In [ ]:
different_label = results["reference_label"] != results["predicted_label"]
model_errors = results.loc[results["status"].eq("ok") & different_label]
print_html(model_errors[["text_id", "text", "reference_label", "predicted_label"]])